# Theorem-Proof adjacency inspector

Validates the Phase 3 ingestion-quality check from PROGRESS.md: for every theorem block (any subkind: theorem, lemma, corollary, proposition) found in a book's chunks, does a proof block follow it inside the same chunk?

For each theorem we report:
- whether the very next block in the same chunk is a proof
- whether a proof appears anywhere later in the same chunk (and at what gap)
- whether the theorem is the last block in its chunk (no chance for adjacency)
- which chunk + page the theorem lives on, so you can cross-check against the PDF

Set `BOOK_ID` to inspect a specific book, or leave it `None` to use the most recently ingested book.

In [10]:
"""Setup: import path, DB session, target book."""
import sys
import uuid
from pathlib import Path

# Make `app.*` importable when running from backend/tests/notebooks/
BACKEND_DIR = Path.cwd()
while BACKEND_DIR.name != "backend" and BACKEND_DIR.parent != BACKEND_DIR:
    BACKEND_DIR = BACKEND_DIR.parent
if BACKEND_DIR.name != "backend":
    raise RuntimeError("Run this notebook from somewhere inside backend/")
sys.path.insert(0, str(BACKEND_DIR))

from app.db.session import SessionLocal
from app.models.book import Book, BookChunk
from app.models.section import BookSection

# Set to a UUID string to inspect a specific book; None = latest ingested book.
BOOK_ID: str | None = None

db = SessionLocal()

if BOOK_ID is None:
    book = (
        db.query(Book)
        .filter(Book.status == "ingested")
        .order_by(Book.created_at.desc())
        .first()
    )
    if book is None:
        raise RuntimeError("No ingested books in the database.")
else:
    book = db.get(Book, uuid.UUID(BOOK_ID))
    if book is None:
        raise RuntimeError(f"Book {BOOK_ID} not found.")

print(f"Book: {book.title}")
print(f"  id     : {book.id}")
print(f"  status : {book.status}")
print(f"  created: {book.created_at}")

Book: Strang — Linear Algebra (Phase 3/4 fixture)
  id     : 73bee0f0-ff45-48d0-b118-f56a7c56b41a
  status : ingested
  created: 2026-05-16 18:02:25.450005+00:00


## 1. Block-kind distribution

A quick sanity check before drilling into theorems — what kinds of blocks did the LLM extract from this book?

In [11]:
"""Load every chunk for this book in chunk order, then count block kinds."""
from collections import Counter

chunks = (
    db.query(BookChunk)
    .filter(BookChunk.book_id == book.id)
    .order_by(BookChunk.chunk_index)
    .all()
)

print(f"Chunks: {len(chunks)}")

kind_counter: Counter[str] = Counter()
theorem_subkind_counter: Counter[str] = Counter()
total_blocks = 0
for c in chunks:
    for b in (c.blocks or []):
        kind = b.get("kind", "(missing)")
        kind_counter[kind] += 1
        total_blocks += 1
        if kind == "theorem":
            theorem_subkind_counter[b.get("subkind", "theorem")] += 1

print(f"Total blocks: {total_blocks}\n")
print(f"{'kind':<14} {'count':>6}")
print("-" * 22)
for kind, count in kind_counter.most_common():
    print(f"{kind:<14} {count:>6}")

if theorem_subkind_counter:
    print("\nTheorem subkinds:")
    for sub, count in theorem_subkind_counter.most_common():
        print(f"  {sub:<12} {count}")
else:
    print("\n(no theorem blocks in this book)")

Chunks: 106
Total blocks: 681

kind            count
----------------------
paragraph         292
exercise          205
equation          106
heading            35
list               13
example            12
remark              6
theorem             6
definition          5
proof               1

Theorem subkinds:
  proposition  4
  theorem      2


In [3]:
### Clean text

In [12]:
for c in chunks:
    if c.page_start == 13:
        print(c.clean_text or '')
        print("----------------------")

First we have to introduce matrices and vectors and the rules for multiplication. Every matrix has a transpose $A^T$. This matrix has an inverse $A^{-1}$.

- In most cases elimination goes forward without difficulties. The matrix has an inverse and the system $Ax = b$ has one solution. In exceptional cases the method will break down—either the equations were written in the wrong order, which is easily fixed by exchanging them, or the equations don't have a unique solution. That **singular** case will appear if 8 replaces 5 in our example:

$$
\begin{aligned} \text{Singular case} \quad 1x + 2y &= 3 \\ \text{Two parallel lines} \quad 4x + 8y &= 6. \end{aligned} \tag{(7)}
$$

Elimination still innocently subtracts 4 times the first equation from the second. But look at the result!

$$
(\text{equation 2}) - 4(\text{equation 1}) \quad 0 = -6.
$$

This singular case has **no solution**. Other singular cases have **infinitely many solutions**. (Change 6 to 12 in the example, and elimination w

## 2. Theorem ↔ Proof adjacency

For every `theorem` block, walk forward in the **same chunk** and classify what we find:

- `IMMEDIATE` — the very next block is a `proof` (textbook ideal)
- `LATER` — a `proof` exists later in the chunk, separated by N other blocks (still acceptable, but a remark/equation got in the middle)
- `MISSING` — no `proof` block anywhere later in this chunk; the theorem is either stated without a proof, or the proof was split into a different chunk

The last case (`MISSING`) is the one that risks breaking the atomic-block adjacency rule downstream. We need to know whether it's `MISSING` because the theorem is genuinely unproved in the source, or because the chunker split the pair across chunk boundaries.

In [3]:
"""Walk every chunk, find every theorem, classify proof adjacency."""
from dataclasses import dataclass

@dataclass
class TheoremHit:
    chunk_index: int
    chunk_id: uuid.UUID
    page: int
    section_title: str | None
    block_index: int
    subkind: str
    label: str | None
    text_preview: str
    status: str           # IMMEDIATE | LATER | MISSING | LAST_BLOCK
    gap: int | None       # blocks between theorem and proof (0 = adjacent), None if no proof
    proof_block_index: int | None
    proof_preview: str | None

def _preview(s: str | None, n: int = 110) -> str:
    if not s:
        return ""
    s = " ".join(s.split())
    return s if len(s) <= n else s[:n] + "..."

hits: list[TheoremHit] = []
for c in chunks:
    blks = c.blocks or []
    for i, b in enumerate(blks):
        if b.get("kind") != "theorem":
            continue
        # Find a proof block AFTER index i in this same chunk
        proof_idx: int | None = None
        for j in range(i + 1, len(blks)):
            if blks[j].get("kind") == "proof":
                proof_idx = j
                break

        if i == len(blks) - 1:
            status = "LAST_BLOCK"
            gap = None
            proof_preview = None
        elif proof_idx is None:
            status = "MISSING"
            gap = None
            proof_preview = None
        elif proof_idx == i + 1:
            status = "IMMEDIATE"
            gap = 0
            proof_preview = _preview(blks[proof_idx].get("markdown"))
        else:
            status = "LATER"
            gap = proof_idx - i - 1
            proof_preview = _preview(blks[proof_idx].get("markdown"))

        hits.append(TheoremHit(
            chunk_index=c.chunk_index,
            chunk_id=c.id,
            page=c.page_start,
            section_title=c.section_title,
            block_index=i,
            subkind=b.get("subkind", "theorem"),
            label=b.get("label"),
            text_preview=_preview(b.get("markdown")),
            status=status,
            gap=gap,
            proof_block_index=proof_idx,
            proof_preview=proof_preview,
        ))

# Summary
status_counter: Counter[str] = Counter(h.status for h in hits)
print(f"Total theorem-like blocks: {len(hits)}")
if hits:
    print("Status breakdown:")
    for st in ("IMMEDIATE", "LATER", "MISSING", "LAST_BLOCK"):
        n = status_counter.get(st, 0)
        pct = (n / len(hits) * 100) if hits else 0
        print(f"  {st:<11} {n:>3}  ({pct:5.1f}%)")

Total theorem-like blocks: 0


## 3. Per-theorem detail

One row per theorem, with section, page, label, status, and a preview of both the theorem statement and (when found) the proof.

In [4]:
"""Per-theorem detail table."""
if not hits:
    print("(no theorems found in this book — nothing to inspect)")
else:
    for h in hits:
        header = f"[{h.subkind.upper()}]  {h.label or '(no label)'}"
        section = h.section_title or "(orphan)"
        gap_str = "" if h.gap is None else f"  gap={h.gap}"
        print(f"\nchunk {h.chunk_index} · page {h.page} · {section}")
        print(f"  {header}   status={h.status}{gap_str}")
        print(f"  theorem: {h.text_preview}")
        if h.proof_preview is not None:
            print(f"  proof  : {h.proof_preview}")
        elif h.status == "LAST_BLOCK":
            print(f"  proof  : (theorem is final block of chunk — possible cross-chunk split)")
        else:
            print(f"  proof  : (none in same chunk)")

(no theorems found in this book — nothing to inspect)


## 4. Investigate "LAST_BLOCK" and "MISSING" cases

A theorem that ends a chunk could mean the proof was pushed to the next chunk. Compare the last block of the theorem's chunk with the first block of the *next* chunk to find out.

In [5]:
"""For each problematic theorem, peek at the next chunk's first block."""
chunks_by_index = {c.chunk_index: c for c in chunks}

problematic = [h for h in hits if h.status in ("LAST_BLOCK", "MISSING")]
if not problematic:
    print("No LAST_BLOCK or MISSING theorems — every theorem has a proof in its own chunk.")
else:
    for h in problematic:
        next_chunk = chunks_by_index.get(h.chunk_index + 1)
        print(f"\nchunk {h.chunk_index} (page {h.page}) — {h.label or h.subkind} [{h.status}]")
        print(f"  theorem: {h.text_preview}")
        if next_chunk is None:
            print("  next chunk: (none — last chunk of book)")
            continue
        next_blocks = next_chunk.blocks or []
        if not next_blocks:
            print(f"  next chunk {next_chunk.chunk_index}: (empty)")
            continue
        first = next_blocks[0]
        first_kind = first.get("kind")
        first_text = first.get("markdown") or first.get("text") or first.get("latex") or ""
        verdict = "PROOF (cross-chunk split)" if first_kind == "proof" else f"{first_kind} (not a proof)"
        print(f"  next chunk {next_chunk.chunk_index} starts with: {verdict}")
        print(f"    {_preview(first_text)}")

No LAST_BLOCK or MISSING theorems — every theorem has a proof in its own chunk.


## 5. Spot-check: pretty-print one theorem-proof pair in full

Set `INSPECT_HIT_INDEX` to the row number from cell 7 (0-indexed) to dump the full markdown of that theorem and (if present) its proof. Useful for eyeballing math rendering and confirming the proof actually proves the theorem.

In [6]:
from IPython.display import Markdown, display

INSPECT_HIT_INDEX = 0

if not hits:
    print("(no theorems to inspect)")
elif INSPECT_HIT_INDEX >= len(hits):
    print(f"INSPECT_HIT_INDEX={INSPECT_HIT_INDEX} out of range (0..{len(hits)-1})")
else:
    h = hits[INSPECT_HIT_INDEX]
    chunk = next(c for c in chunks if c.id == h.chunk_id)
    blks = chunk.blocks or []

    md_parts = [
        f"### {h.subkind.title()} — {h.label or '(no label)'}",
        f"_chunk {h.chunk_index} · page {h.page} · {h.section_title or 'orphan'} · status `{h.status}`_",
        "",
        "**Statement**",
        "",
        blks[h.block_index].get("markdown", "(empty)"),
    ]
    if h.proof_block_index is not None:
        md_parts += [
            "",
            f"**Proof** _(block {h.proof_block_index}, gap={h.gap})_",
            "",
            blks[h.proof_block_index].get("markdown", "(empty)"),
        ]
    else:
        md_parts += ["", "_(no proof block in this chunk)_"]

    display(Markdown("\n".join(md_parts)))

(no theorems to inspect)
